Check if HLA extraction have been done for all AGD 250k samples on ICA.

Save commands for failed ones to re-run.

In [4]:
import pandas as pd
import os
import subprocess
import gzip
import glob

## 1. Check existing samples by bam files, log files against our vcf file

In [87]:
# 1. Load samples from The AGD VCF on the server
print('# Load samples from The AGD VCF on the server')
# fn_vcf = '/data100t1/share/BioVU/agd_250k/primary-pass-pgen/agd250k_chr22.primary_pass.psam'
# df_samples_in_vcf = pd.read_csv(fn_vcf, sep='\t').iloc[10:, :] # Skip the first 10 as they are controls
# df_samples_in_vcf.head()
# Load from the vcf instead of the plink files, they are the same
fn_vcf = '/data100t1/share/BioVU/agd_250k/primary-pass-msvcf/agd250k_chr22.primary_pass.vcf.gz'
with gzip.open(fn_vcf, 'rt') as fh:
    for line in fh:
        if line.startswith('##'):
            continue
        else: break

df_samples_in_vcf = pd.DataFrame({'IID':line.strip().split('\t')[19:]})


# 2. Get samples by existing bam files
print('# Get samples by existing bam files')
path_bams = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
c = 0
lst_samples_with_bam = []
for i in range(1, 250002, 5000):
    c += 1
    sample_range = f'{i}-{i+5000-1}'
    bam_folder = f'{path_bams}/workspace_extraction_{sample_range}'
    for fn in os.listdir(bam_folder):
        # sample_id = fn.split('.')[0]
        lst_samples_with_bam.append(fn.split('.')[0])

lst_samples_with_bam = list(set(lst_samples_with_bam))
print('# -', len(lst_samples_with_bam))

# 3. Get samples by log files
print('# Get samples by log files')
path_logs = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams/ica_logs'
lst_samples_with_log = []
for fn in os.listdir(path_logs):
    with open(f'{path_logs}/{fn}') as fh:
        for line in fh:
            lst_samples_with_log.append(os.path.split(line)[-1].split('.')[0])

lst_samples_with_log = list(set(lst_samples_with_log))
print('# -', len(lst_samples_with_log))

print('# Get samples differences')
# Find samples not in the log file or without the bam file
df_samples_no_bam = df_samples_in_vcf[~df_samples_in_vcf['IID'].isin(lst_samples_with_bam)]
df_samples_not_in_log = df_samples_in_vcf[~df_samples_in_vcf['IID'].isin(lst_samples_with_log)]

print('# - N samples without a bam file:', len(df_samples_no_bam))
print('# - N samples not in log files:', len(df_samples_not_in_log))


# Load samples from The AGD VCF on the server
# Get samples by existing bam files
# - 249801
# Get samples by log files
# - 250105
# Get samples differences
# - N samples without a bam file: 531
# - N samples not in log files: 227


In [106]:
# Create a re-run on samples without bams or not logged
df_all_samples = pd.concat([df_samples_no_bam, df_samples_not_in_log])
df_all_samples['IID_reformat'] = df_all_samples['IID'].apply(lambda x: x.split('_')[0])
print(len(df_all_samples))

# Remove duplicates
df_all_samples.drop_duplicates(subset='IID_reformat', inplace=True)
print('# Total number of samples need rerun:', len(df_all_samples))

758
# Total number of samples need rerun: 531


# 2. Create commands for re-run
Since the sample size is small, I can use the CLI to submit pipeline runs

In [ ]:
# Get the sample file IDs on ICA
fn_file_ids = '/DC3/AGD250k_CRAM_info_only/agd_cram_file_ids_from_2025_q1_release.na_fixed.tsv'
df_file_ids = pd.read_csv(fn_file_ids, sep='\t')
df_file_ids

df_rerun_samples = df_file_ids[df_file_ids['sample_id'].isin(df_all_samples['IID_reformat'])]

# # Create CLI commends
# for i, row in df_rerun_samples.iterrows():
#     sample_id, cram_id, cram_index_id = row
#     cmd = f'pipeline_id=156855c9-d3c3-4a64-9768-5a45d0b7f027; '
#     cmd += 'output_parent_folder_id=fol.52f00bf314b74540d1a508dec56d8214; ' # /wanying/kourami_hla_typing/chr6_hla_extraction/20260608_rerun/
#     cmd += f'cram_file_id={cram_id}; '
#     cmd += f'crai_file_id={cram_index_id}; '
#     cmd += 'cram_ref_file_id=fil.d0a8266ac310465a5fff08deaaf769c0; '
#     cmd += 'icav2 projectpipelines start nextflow ${pipeline_id} '
#     cmd += '--input cram_path:${cram_file_id} '
#     cmd += '--input cram_index:${crai_file_id} '
#     cmd += '--input cram_ref:${cram_ref_file_id} '
#     cmd += '--output-parent-folder ${output_parent_folder_id} '
#     cmd += '--storage-size small '
#     cmd += f"--user-reference 'rerun_nextflow_pipeline_via_CLI_1_sample'"
c = 0
for i 


In [142]:
cram_ref_file_id = 'fil.d0a8266ac310465a5fff08deaaf769c0'
output_fn = './bash_slurm/ica_rerun_pipelines.sh'
output_fh = open(output_fn, 'w')
log_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams/rerun_failed_ones'
n = 10 #chunk row size
c = 0
for df in [df_rerun_samples[i:i+n] for i in range(0, df_rerun_samples.shape[0], n)]:
    c += 1
    cmd = f'pipeline_id=156855c9-d3c3-4a64-9768-5a45d0b7f027; '
    cmd += 'output_parent_folder_id=fol.52f00bf314b74540d1a508dec56d8214; ' # /wanying/kourami_hla_typing/chr6_hla_extraction/20260608_rerun/
    cram_ids = '"' + '","'.join(df['cram_id']) + '"'
    cram_index_ids = '"' + '","'.join(df['cram_index_id']) + '"'
    cmd += 'icav2 projectpipelines start nextflow ${pipeline_id} '
    cmd += f"--input cram_path:{cram_ids} "
    cmd += f"--input cram_index:{cram_index_ids} "
    cmd += f'--input cram_ref:{cram_ref_file_id} '
    cmd += '--output-parent-folder ${output_parent_folder_id} '
    cmd += '--storage-size small '
    cmd += f"--user-reference 'rerun_nextflow_pipeline_{c}' > {log_path}/rerun_nextflow_pipeline_{c}.log"
    output_fh.write(cmd + '\n')
output_fh.close()


## !! There is a small number of people do not have CRAM on the ICA

I manually checked a few on the ICA

In [112]:
df_missing_cram = df_all_samples[~df_all_samples['IID_reformat'].isin(df_file_ids['sample_id'])].sort_values('IID')

print(len(df_missing_cram))
df_missing_cram

49


,IID,IID_reformat
13341,R203742081,R203742081
127115,R205023624,R205023624
66617,R207321914,R207321914
186396,R207574593,R207574593
53977,R208330629,R208330629
59783,R210069680,R210069680
25250,R212832085,R212832085
242284,R213473485,R213473485
132825,R213707061,R213707061
84458,R213841708,R213841708


# 3. Get re-run samples from ica
Those samples are extracted reads sorted by samtools, but should not matter if I sort them again.

In [ ]:
# Get folder information of each pipeline run from the submission logs
path_logs = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams/rerun_failed_ones/logs'
ica_parent_folder = '/wanying/kourami_hla_typing/chr6_hla_extraction/20260608_rerun'
dict_result_info_on_ica = dict()
output_fh = open('./bash_slurm/download_rerun_bams.sh', 'w')
target_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams/rerun_failed_ones/'
for fn in os.listdir(path_logs):
    with open(f'{path_logs}/{fn}') as fh:
        for line in fh:
            key, val = line.strip().split(maxsplit=1)
            dict_result_info_on_ica[key] = val
    result_folder = f"{ica_parent_folder}/{dict_result_info_on_ica['userReference']}-{dict_result_info_on_ica['id']}"
    # Get path to the bams
    cmd = f"/data100t1/home/wanying/downloaded_tools/icav2/icav2 projectdata list --parent-folder  {result_folder}/"
    cmd += ' --file-name .bam --match-mode FUZZY'
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout
    file_paths = [val.split()[-1] for val in res.strip().split('\n')[1:-1]]

    # Create commands to download the bams
    for file_path in file_paths:
        cmd = f'/data100t1/home/wanying/downloaded_tools/icav2/icav2 projectdata download {file_path} {target_path}'
        output_fh.write(cmd + '\n')
output_fh.close()


# 4. Create a file of sample id and bam path

In [9]:
bam_path = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams'
lst_bam_folder = [f'{bam_path}/workspace_extraction_{i}-{i+5000-1}' for i in range(1, 250002, 5000)] + [f'{bam_path}/rerun_failed_ones']
c_total = 0
lst_sample_ids, lst_bam_path = [], []

for path in lst_bam_folder:
    for bam_path in glob.glob(f'{path}/*bam'):
        sample_id = bam_path.split('/')[-1].split('.')[0]
        lst_sample_ids.append(sample_id)
        lst_bam_path.append(bam_path)
df_bam_path = pd.DataFrame({'grid':lst_sample_ids, 'bam_path':lst_bam_path})
output_fn = '/DC3/AGD250k_HLA_typing/extracted_hla_read_bams/agd250k_hla_bam_paths.txt'
df_bam_path.to_csv(output_fn, index=False, sep='\t')

output_fn = '../data/agd250k_hla_bam_paths.txt'
df_bam_path.to_csv(output_fn, index=False, sep='\t')